[< Back to Demo 01b](../01-graphrag-demo/retrieval_patterns.ipynb) | [Demo README](README.md)

# Optional Demo 09: Neo4j MCP and Controlled Text2Cypher

Demo 01b ran Text2Cypher in the notebook process with direct Neo4j credentials. This module reaches the same graph through a pre-deployed Neo4j MCP service. The capability is still read Cypher; the difference is where credentials, query classification, timeout, and response truncation are enforced.

The participant path has three beats: fail-closed tool discovery, one reviewed parameterized template, and one optional Text2Cypher question. It never attempts a write to prove the boundary.

## Configuration

Set `NEO4J_MCP_URL` and, when required, `NEO4J_MCP_TOKEN` in the environment or a `.env` file next to this notebook. The operator aliases `MCP_GATEWAY_URL` and `MCP_ACCESS_TOKEN` are also accepted. The notebook does not read the old workshop's `CONFIG.txt`.

The reviewed template needs only MCP access. The optional Text2Cypher cell also needs AWS credentials, `AWS_REGION`, and Bedrock access to the pinned workshop model. Every MCP operation uses its own context-managed connection, so failures cannot leave a session open.

In [ ]:
import os
import uuid

import boto3
from dotenv import load_dotenv
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp import MCPClient

from mcp_config import (
    SKIP_MESSAGE,
    load_mcp_settings,
    require_read_tools,
    tool_result_records,
    tool_result_text,
)

load_dotenv()
settings = load_mcp_settings()
MCP_READY = settings.configured
BEDROCK_READY = boto3.Session().get_credentials() is not None
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_ID = os.getenv("MODEL_ID", "us.anthropic.claude-sonnet-4-6")

def new_mcp_client():
    return MCPClient(
        lambda: streamablehttp_client(
            url=settings.url,
            headers=settings.headers() or None,
        )
    )

def approved_tools(client):
    return require_read_tools(client.list_tools_sync())

if MCP_READY:
    auth = "bearer token set" if settings.token else "tokenless"
    print(f"MCP endpoint configured: {settings.url} ({auth})")
else:
    print(SKIP_MESSAGE)

## 1. Discover exactly two approved read tools

The notebook accepts only `get_neo4j_schema` and `read_neo4j_cypher`. A missing tool or any unexpected tool, including a write tool, raises before an agent is built. Only the approved wrappers are ever passed to the model.

In [ ]:
if not MCP_READY:
    print("Skipping tool discovery: no endpoint configured.")
else:
    with new_mcp_client() as client:
        tools = approved_tools(client)
        print("Approved MCP tools:")
        for tool in tools:
            print(f"- {tool.tool_name}")

## 2. Inspect the live schema server-side

The official server introspects the current database schema when this tool is called. This is live server-side discovery, not a filtered or pinned schema view. In a production deployment, expose a governed schema tool if the model should see only selected labels and relationships.

In [ ]:
if not MCP_READY:
    print("Skipping schema discovery: no endpoint configured.")
else:
    with new_mcp_client() as client:
        approved_tools(client)
        result = client.call_tool_sync(
            tool_use_id=f"schema-{uuid.uuid4()}",
            name="get_neo4j_schema",
            arguments={},
        )
    if result.get("status") != "success":
        raise RuntimeError(tool_result_text(result) or "schema tool failed")
    schema_text = tool_result_text(result)
    print(schema_text[:2500] + ("…" if len(schema_text) > 2500 else ""))

## 3. Run one reviewed template

A pinned template is the strongest control for a known question shape. The model cannot change its labels, traversal, return columns, or `LIMIT`; the hotel name is a query parameter rather than string concatenation.

In [ ]:
AMENITIES_TEMPLATE = """
CYPHER 25
MATCH (hotel:Hotel)
WHERE toLower(hotel.name) CONTAINS toLower($hotel_name)
OPTIONAL MATCH (hotel)-[:OFFERS_AMENITY]->(amenity:Amenity)
RETURN hotel.name AS hotel,
       hotel.guest_rating AS guest_rating,
       collect(DISTINCT amenity.name)[..12] AS amenities
ORDER BY hotel
LIMIT 5
"""
TEMPLATE_PARAMS = {"hotel_name": "AnyCompany Cairo Nile View"}
print(AMENITIES_TEMPLATE)
print(f"Parameters: {TEMPLATE_PARAMS}")

In [ ]:
if not MCP_READY:
    print("Skipping template: no endpoint configured.")
else:
    with new_mcp_client() as client:
        approved_tools(client)
        result = client.call_tool_sync(
            tool_use_id=f"template-{uuid.uuid4()}",
            name="read_neo4j_cypher",
            arguments={"query": AMENITIES_TEMPLATE, "params": TEMPLATE_PARAMS},
        )
    records = tool_result_records(result)
    for record in records:
        amenities = ", ".join(
            item for item in (record.get("amenities") or []) if item
        )
        print(f"{record.get('hotel')}: {amenities or 'none recorded'}")

## 4. Optional controlled Text2Cypher

This optional step passes only the two approved tools to a Strands agent. The prompt asks for schema-first, read-only, parameterized, limited Cypher. Prompt instructions improve behavior, while the meaningful enforcement remains outside the model: least-privilege database credentials, server read-query classification, query timeout, and response truncation. The response-token cap is not a database row limit.

In [ ]:
if not MCP_READY:
    print("Skipping Text2Cypher: no endpoint configured.")
elif not BEDROCK_READY:
    print("Skipping Text2Cypher: AWS credentials are not configured.")
else:
    from strands import Agent
    from strands.models import BedrockModel

    system_prompt = """You are a hotel knowledge graph assistant.
Always inspect the schema first. Generate only Cypher 25 read queries, use parameters
for dynamic values, include a LIMIT, and answer only from query results.
"""
    with new_mcp_client() as client:
        tools = approved_tools(client)
        agent = Agent(
            model=BedrockModel(
                model_id=MODEL_ID,
                region_name=AWS_REGION,
                temperature=0,
            ),
            system_prompt=system_prompt,
            tools=tools,
        )
        agent("How many hotels offer a swimming pool?")

## Why there is no live write-rejection probe

A safety demo should not attempt `CREATE` against a live graph. If the endpoint were misconfigured, that proof would mutate the database and leave an orphan node. The event preflight validates write rejection in a disposable operator environment. Participant safety comes from the exact tool allowlist, a server configured for read-only queries, and a database account with read-only privileges.

## When to use each path

| Situation | Prefer |
|-----------|--------|
| Known repeated question with a stable result contract | Reviewed template |
| Compliance review of every query | Reviewed template |
| Ad hoc exploration and long-tail questions | Controlled Text2Cypher |
| Production hot path in this workshop | Demo 06A's fixed Hybrid-Cypher retrieval |

The MCP read tool intentionally supports arbitrary read Cypher. Use it for governed exploration; use a finite template catalog when the question space is known.